In [ ]:
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from scipy.stats import pearsonr
import matplotlib.pyplot as plt

# --- 1. Load both datasets ---
news = pd.read_csv('data/raw/news.csv')
stocks = pd.read_csv('data/raw/AAPL_stock.csv', parse_dates=['Date'])

# --- 2. Score each headline with VADER ---
analyzer = SentimentIntensityAnalyzer()
news['sentiment'] = news['headline'].apply(
    lambda x: analyzer.polarity_scores(x)['compound']  # -1 to +1
)

# --- 3. Date Alignment ---
news['date'] = pd.to_datetime(news['date'], utc=True).dt.tz_localize(None)
news['date_only'] = news['date'].dt.normalize()

# If published on weekend → shift to next Monday
import numpy as np
news['date_only'] = news['date_only'].apply(
    lambda d: d + pd.offsets.BDay(1) if d.weekday() >= 5 else d
)

# Aggregate: average sentiment per stock per day
daily_sentiment = news.groupby(['date_only', 'stock'])['sentiment'].mean().reset_index()

# --- 4. Calculate daily stock returns ---
stocks = stocks.sort_values('Date')
stocks['daily_return'] = stocks['Adj Close'].pct_change() * 100

# --- 5. Merge and Correlate ---
merged = pd.merge(
    daily_sentiment, stocks[['Date', 'daily_return']],
    left_on='date_only', right_on='Date'
)
merged.dropna(inplace=True)

corr, pvalue = pearsonr(merged['sentiment'], merged['daily_return'])
print(f"Pearson Correlation: {corr:.4f}, p-value: {pvalue:.4f}")

# Scatter plot
plt.figure(figsize=(8, 5))
plt.scatter(merged['sentiment'], merged['daily_return'], alpha=0.5)
plt.xlabel('Average Daily Sentiment')
plt.ylabel('Daily Return (%)')
plt.title(f'Sentiment vs Stock Return (r = {corr:.3f})')
plt.axhline(0, color='gray', linestyle='--')
plt.axvline(0, color='gray', linestyle='--')
plt.savefig('notebooks/plots/sentiment_correlation.png')
plt.show()

# --- 6. Bar chart by sentiment category ---
merged['sentiment_label'] = pd.cut(
    merged['sentiment'], bins=[-1, -0.05, 0.05, 1],
    labels=['Negative', 'Neutral', 'Positive']
)
merged.groupby('sentiment_label')['daily_return'].mean().plot(kind='bar', color=['red','gray','green'])
plt.title('Average Daily Return by Sentiment Category')
plt.ylabel('Avg Return (%)')
plt.show()

In [ ]:
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from scipy.stats import pearsonr
import matplotlib.pyplot as plt

# --- 1. Load both datasets ---
news = pd.read_csv('data/raw/news.csv')
stocks = pd.read_csv('data/raw/AAPL_stock.csv', parse_dates=['Date'])

# --- 2. Score each headline with VADER ---
analyzer = SentimentIntensityAnalyzer()
news['sentiment'] = news['headline'].apply(
    lambda x: analyzer.polarity_scores(x)['compound']  # -1 to +1
)

# --- 3. Date Alignment ---
news['date'] = pd.to_datetime(news['date'], utc=True).dt.tz_localize(None)
news['date_only'] = news['date'].dt.normalize()

# If published on weekend → shift to next Monday
import numpy as np
news['date_only'] = news['date_only'].apply(
    lambda d: d + pd.offsets.BDay(1) if d.weekday() >= 5 else d
)

# Aggregate: average sentiment per stock per day
daily_sentiment = news.groupby(['date_only', 'stock'])['sentiment'].mean().reset_index()

# --- 4. Calculate daily stock returns ---
stocks = stocks.sort_values('Date')
stocks['daily_return'] = stocks['Adj Close'].pct_change() * 100

# --- 5. Merge and Correlate ---
merged = pd.merge(
    daily_sentiment, stocks[['Date', 'daily_return']],
    left_on='date_only', right_on='Date'
)
merged.dropna(inplace=True)

corr, pvalue = pearsonr(merged['sentiment'], merged['daily_return'])
print(f"Pearson Correlation: {corr:.4f}, p-value: {pvalue:.4f}")

# Scatter plot
plt.figure(figsize=(8, 5))
plt.scatter(merged['sentiment'], merged['daily_return'], alpha=0.5)
plt.xlabel('Average Daily Sentiment')
plt.ylabel('Daily Return (%)')
plt.title(f'Sentiment vs Stock Return (r = {corr:.3f})')
plt.axhline(0, color='gray', linestyle='--')
plt.axvline(0, color='gray', linestyle='--')
plt.savefig('notebooks/plots/sentiment_correlation.png')
plt.show()

# --- 6. Bar chart by sentiment category ---
merged['sentiment_label'] = pd.cut(
    merged['sentiment'], bins=[-1, -0.05, 0.05, 1],
    labels=['Negative', 'Neutral', 'Positive']
)
merged.groupby('sentiment_label')['daily_return'].mean().plot(kind='bar', color=['red','gray','green'])
plt.title('Average Daily Return by Sentiment Category')
plt.ylabel('Avg Return (%)')
plt.show()